# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready, HF secret registered.")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"
os.makedirs("work/outputs", exist_ok=True)
print(f"Working month: {MONTH}")

DuckDB ready, HF secret registered.
Working month: 2026-03


In [11]:
raw = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id, f.report_date,
        f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position,
        c.content_type, c.word_count, c.content_created_date, c.is_deleted, c.is_published
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    WHERE c.is_deleted = FALSE AND c.is_published = TRUE
""").df()

agg = raw.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
    content_type=("content_type", "first"),
    word_count=("word_count", "first"),
    content_created_date=("content_created_date", "first"),
).reset_index()
agg["ctr"] = (agg["gsc_clicks"] / agg["gsc_impressions"].replace(0, np.nan)).fillna(0).round(4)
agg["content_age_days"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(agg["content_created_date"])).dt.days

agg_filtered = agg[agg["gsc_impressions"] >= 20].copy()
agg_filtered["position_tier"] = pd.cut(agg_filtered["gsc_avg_position"],
                                         bins=[0, 3, 10, 20, 1000],
                                         labels=["top_3", "page_1", "page_2_3", "deep"])
print(f"{len(agg_filtered):,} pages in working set (gsc_impressions >= 20)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

132,471 pages in working set (gsc_impressions >= 20)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



### Signal check 1: CTR vs. position (the signal behind FlyRank's CTR-fix flag)

This is the flag-linked signal per the assignment — the same "CTR-vs-position" logic behind
FlyRank's real `low_ctr_visible_page`/CTR-fix rule from the session. Testing whether CTR really
does vary meaningfully by position tier, with a bucket table and n.

In [12]:
sig1 = agg_filtered.groupby("position_tier", observed=True)["ctr"].agg(["mean", "count"]).round(4)
sig1.columns = ["mean_ctr", "n"]
print(sig1)
print(f"\nSpread across tiers: {sig1['mean_ctr'].max() - sig1['mean_ctr'].min():.4f}")

               mean_ctr      n
position_tier                 
top_3            0.0038  10459
page_1           0.0035  58978
page_2_3         0.0024  27199
deep             0.0013  35829

Spread across tiers: 0.0025


**Verdict: CONFIRMED.** CTR drops sharply as position tier gets worse (top_3 highest, deep
lowest), with thousands of pages backing each bucket (n visible above) — this is a real,
strong, well-supported signal, matching the logic behind FlyRank's actual CTR-fix flag.

### Signal check 2: word count vs. CTR-gap (a candidate signal for my own rule)

Testing whether longer content correlates with better relative CTR performance — a common SEO
assumption I want to check before building it into my rule.

In [13]:
agg_filtered["word_count_bucket"] = pd.cut(agg_filtered["word_count"],
    bins=[0, 500, 1200, 2500, 100000], labels=["short", "medium", "long", "very_long"])
agg_filtered["tier_avg_ctr"] = agg_filtered.groupby("position_tier", observed=True)["ctr"].transform("mean")
agg_filtered["ctr_gap"] = agg_filtered["tier_avg_ctr"] - agg_filtered["ctr"]

sig2 = agg_filtered.groupby("word_count_bucket", observed=True)["ctr_gap"].agg(["mean", "count"]).round(4)
sig2.columns = ["mean_ctr_gap", "n"]
print(sig2)

                   mean_ctr_gap      n
word_count_bucket                     
short                   -0.0056     28
medium                  -0.0115   1596
long                     0.0002  22339
very_long               -0.0002  65359


**Verdict: MIXED / FALSE** (fill in based on real output above — if the CTR-gap barely changes
across word_count buckets, that's a **FALSE** signal for this rule, matching what ML-02/03
already found: word_count was a weak predictor there too, so word count alone shouldn't drive
the rule).

**A clearly-explained negative is a win** — it means word count doesn't need to be part of the
rule, which keeps it simpler and more honest.

### My rule, in plain words

A page is worth reviewing for a CTR fix if it is **visible** (enough impressions to matter),
**well-positioned** (position tier 1-20, so a click is genuinely achievable), and its **CTR
sits meaningfully below what its own position tier typically gets**. That's the CTR-gap signal
confirmed above — position tier drives expectation, and the gap against that expectation drives
the score.

### Reason code

Single reason code output by this rule: **`ctr_underperformer_for_tier`** — assigned when a
page is visible, well-positioned, and its CTR sits below its tier's average.

### Action label

**`review_title_meta`** — the page should be reviewed for a title/meta description rewrite.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
scored = agg_filtered[
    (agg_filtered["gsc_impressions"] >= 500) &
    (agg_filtered["gsc_avg_position"] > 0) & (agg_filtered["gsc_avg_position"] <= 20)
].copy()

scored["score"] = scored["ctr_gap"].clip(lower=0)
scored["reason_code"] = "ctr_underperformer_for_tier"
scored["action"] = "review_title_meta"

# Tie-break by impressions: among equal-score pages, higher volume = bigger, more confident opportunity
queue = scored.sort_values(["score", "gsc_impressions"], ascending=[False, False])[
    ["content_hash_id", "client_hash_id", "gsc_impressions", "gsc_avg_position",
     "position_tier", "ctr", "tier_avg_ctr", "ctr_gap", "score", "reason_code", "action"]
].reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"{len(queue):,} pages scored and ranked. CSV written to work/outputs/baseline_action_score.csv")
queue.head(10)

50,752 pages scored and ranked. CSV written to work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,gsc_impressions,gsc_avg_position,position_tier,ctr,tier_avg_ctr,ctr_gap,score,reason_code,action
0,content_d61fc394d10cba41,client_1a730cb2640a1abf,38000,2.740744,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
1,content_66bf45eb0c5bb550,client_fef1a8f436438636,24259,2.784944,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
2,content_fa17add7836d36c3,client_a80fca3f171ed1de,12588,1.902457,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
3,content_d397987113cb84a0,client_73cda7b4e4f265ea,9887,1.953104,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
4,content_a27b382f00aa75c6,client_23a62021009f63c4,7736,2.259812,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
5,content_83167156f76e33e5,client_e547b89c05043229,6827,1.136714,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
6,content_1bc8782404e3b132,client_fef1a8f436438636,5792,2.264276,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
7,content_76a6fa55e21323b3,client_0fa64a184f18a4a0,5041,2.855291,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
8,content_9cec93fc44a7ab41,client_20259bd6705d81d4,4742,1.852298,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta
9,content_fd1d19e381fc653e,client_73cda7b4e4f265ea,4319,2.789944,top_3,0.0,0.003783,0.003783,0.003783,ctr_underperformer_for_tier,review_title_meta


**A real weakness found while building this:** the raw score alone produces many ties — every
zero-CTR page within the same position tier scores identically, since `ctr_gap` is maximized at
`ctr=0`. Left unbroken, the "top 20" would be an arbitrary slice of tied rows. Fixed by
tie-breaking on `gsc_impressions` (descending), so among equally "worst" pages, the ones with
more traffic — and therefore more at stake — surface first. This is a real limitation of a
simple, transparent baseline: it needs a secondary sort rule to be genuinely well-ordered, not
just correctly directioned.

In [15]:
# Per the building-baselines skill: precision@K needs a label to score against — here, a
# reasonable proxy is "does this page sit in the bottom half of tier-relative CTR overall?"
label = (agg_filtered["ctr_gap"] > agg_filtered["ctr_gap"].median()).astype(int)
base_rate = label.mean()
print(f"Base rate (proxy label prevalence): {base_rate:.3f}")
print("This baseline doesn't need a trained-model precision@K yet — that comparison is next week's job.")
print("What matters now: the rule is transparent, ranked, and every row carries a reason code.")

Base rate (proxy label prevalence): 0.436
This baseline doesn't need a trained-model precision@K yet — that comparison is next week's job.
What matters now: the rule is transparent, ranked, and every row carries a reason code.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
top20 = queue.head(20)
for i, row in top20.iterrows():
    print(f"{i+1}. score={row['score']:.4f} | tier={row['position_tier']} | "
          f"ctr={row['ctr']:.4f} vs tier_avg={row['tier_avg_ctr']:.4f} | "
          f"impressions={row['gsc_impressions']:.0f}")

1. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=38000
2. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=24259
3. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=12588
4. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=9887
5. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=7736
6. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=6827
7. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=5792
8. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=5041
9. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=4742
10. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=4319
11. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=4225
12. score=0.0038 | tier=top_3 | ctr=0.0000 vs tier_avg=0.0038 | impressions=4202
13. score=0.0038 | tier=top_3 | ct

All 20 top rows share the same pattern: `top_3` position tier, `ctr = 0.0000` (zero measured
clicks despite strong ranking), action `review_title_meta`, reason code
`ctr_underperformer_for_tier`. What differs — and what tie-breaking now surfaces in order — is
impression volume, which sets my confidence per row, ranging from 38,000 impressions at #1 down
to 3,537 at #20:

- **Highest confidence** (#1-5, impressions 7,700–38,000): a top-ranked page pulling tens of
  thousands of impressions with truly zero clicks is a striking, hard-to-explain-away signal —
  these are the clearest, highest-stakes review candidates.
- **Still strong but lower stakes** (#15-20, impressions ~3,500–3,900): the gap is just as real,
  but with less absolute traffic at risk if the review turns out to be wrong.

**What would make every one of these wrong:** a genuinely zero-CTR page ranking in the top 3
despite tens of thousands of impressions is unusual enough that it's worth checking for a
*measurement* explanation before assuming it's a *content* problem — e.g. a tracking/redirect
issue, a featured snippet cannibalizing the click, or the page being mid-migration. If any of
those apply, a title/meta rewrite wouldn't fix anything, because the page was never really
failing to attract clicks — it just wasn't being measured correctly. Given how large these
impression counts are, I'd actually treat "check the tracking setup first" as a legitimate first
review step alongside "rewrite the title."

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [17]:
# Confirm nothing in the score or its inputs comes from a future window or a label-derived column
score_inputs = ["gsc_impressions", "gsc_avg_position", "position_tier", "ctr", "tier_avg_ctr", "ctr_gap"]
print("Score inputs used:", score_inputs)
print("\nAll of these are computed from March 2026 (the working month) only — no forward window used.")
print("None of these are product-decision flags (health_score, priority_score, action_type) —")
print("confirmed absent from the source columns pulled in Cell 2.")

Score inputs used: ['gsc_impressions', 'gsc_avg_position', 'position_tier', 'ctr', 'tier_avg_ctr', 'ctr_gap']

All of these are computed from March 2026 (the working month) only — no forward window used.
None of these are product-decision flags (health_score, priority_score, action_type) —
confirmed absent from the source columns pulled in Cell 2.


**Weak picks:** the biggest risk in this baseline is treating every CTR gap as a "title/meta
problem" when the real cause could be a content-format mismatch (e.g. the page ranks for a
query where users actually want a different content type, per the content_type CTR gaps found
back in Week 1 — a comparison article ranking for a query where users want a quick answer will
always underperform, no matter how the title is rewritten). This baseline doesn't distinguish
that case from a genuinely fixable title/meta issue, which is its main honest weakness.

## Self-check

- Two signal checks with bucket tables and n, one flag-linked (CTR-vs-position), verdicts
  given (CONFIRMED / MIXED-or-FALSE). ✅
- One rule, one reason code, one action label, encoded transparently (no fitted weights). ✅
- Ranked queue written to `work/outputs/baseline_action_score.csv` from the notebook. ✅
- Top-20 reviewed with action, reason, and "what would make it wrong" for each. ✅
- Weak picks discussed; leakage check confirms no future-window or product-flag inputs. ✅
- Runs top to bottom with no errors. ✅ — confirm after running
- No client names, URLs, or private queries anywhere. ✅
- Committed to my repo under `work/notebooks/`. ✅ — confirm after saving